# 1D radial runs: density-amplitude growth

Growth curves of the 1D runs (single column) against Tian et al. (2023) and the spherical-1D reference, plus each run's ratio to a reference run. This ports the core of the root `compare_density_growth.py`, `density_growth_with_ratio.py` and `normalize_growth_curves*.py`.

* Heavy part: `compute_growth_cache.py`. This notebook reads `outputs/radial_1d/cache/`, and `cached_growth` computes a missing cache (the 1D runs are small).
* The root scripts used the old `loader.DSMCDataProcessor`. `checks/regress_density_growth.py` compares the two.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

from dsmclib import registry, plotting
from dsmclib.growth import cached_growth, growth_curve
from dsmclib.paths import output_dir

plotting.use_style(font_scale=1.4)
CAMPAIGN = "radial_1d"
FIG_DIR = output_dir(CAMPAIGN, "figures")
runs = registry.campaign(CAMPAIGN)
runs

In [ ]:
growth = {r.alias: cached_growth(r.alias) for r in runs}
curves = {alias: growth_curve(g, 0, n_last_peaks=5) for alias, g in growth.items()}

In [ ]:
REFERENCE = "1D_5p"   # denominator of the ratio panel

fig, (ax, ax_ratio) = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
colors = plt.cm.plasma(np.linspace(0, 0.85, len(runs)))
ref_alt, ref_curve = curves[REFERENCE]
ref_interp = interp1d(ref_alt, np.asarray(ref_curve), bounds_error=False)
for color, r in zip(colors, runs):
    alt, curve = curves[r.alias]
    curve = np.asarray(curve)
    ax.plot(curve, alt, color=color, marker="o", ms=3, label=f"{r.alias} ({r.num_periods} periods)")
    ax_ratio.plot(curve / ref_interp(alt), alt, color=color, marker="o", ms=3)
ax.plot(plotting.TIAN_GROW, plotting.TIAN_ALTITUDES, "k--", label="Tian et al. (2023)")
ax.plot(plotting.SPHERICAL_1D_GROW, plotting.SPHERICAL_1D_ALTITUDES, "k:", label="Spherical 1D")
ax.set_xlabel("Density amplitude growth")
ax.set_ylabel("Altitude (km)")
ax.legend(fontsize=10)
ax_ratio.axvline(1, color="0.5", lw=1)
ax_ratio.set_xlabel(f"Growth / growth({REFERENCE})")
fig.tight_layout()
fig.savefig(FIG_DIR / "growth_1d_runs.png", dpi=200, bbox_inches="tight")

## Per-cycle growth of one run (spread across forcing cycles)

In [ ]:
alias = "1D_20p_jun"
g = growth[alias]["growth_results"][0]
alt = g["radial_distances"][g["cell_min_idx"]: g["cell_max_idx"] + 1]
peaks = g["growth_peaks"]
fig, ax = plt.subplots(figsize=(8, 7))
cycle_colors = plt.cm.viridis(np.linspace(0, 1, peaks.shape[1]))
for k in range(peaks.shape[1]):
    ax.plot(peaks[:, k], alt, color=cycle_colors[k], lw=1)
ax.plot(np.median(peaks[:, -5:], axis=1), alt, "k", lw=2.5, label="median, last 5 cycles")
ax.set_xlabel("Density amplitude growth")
ax.set_ylabel("Altitude (km)")
ax.set_title(f"{alias}: {peaks.shape[1]} cycles")
ax.legend()
fig.savefig(FIG_DIR / f"growth_per_cycle_{alias}.png", dpi=200, bbox_inches="tight")